# Моделирование. Проведение экспериментов. Подготовка пайплайнов обработки данных и построения модели.

# Импорты и настройки

In [42]:
# Стандартная библиотека
import os
import sys
import warnings
import joblib

# Работа с данными
import numpy as np
import pandas as pd

# ML
import lightgbm as lgb
import mlflow

# Настройки
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:,.0f}".format

# Пути проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.config import TRAIN_PATH, DATA_DIR, RAW_DIR

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Проверка путей
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", os.path.exists(TRAIN_PATH))
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("RAW_DIR exists:", os.path.exists(RAW_DIR))
print("MODELS_DIR exists:", os.path.exists(MODELS_DIR))

PROJECT_ROOT: /home/mle_projects/bank-product-recs
TRAIN_PATH exists: True
DATA_DIR exists: True
RAW_DIR exists: True
MODELS_DIR exists: True


# Загрузка данных

In [43]:
df = pd.read_csv(TRAIN_PATH, low_memory=False)
print(df.shape)
df.head()

(13647309, 48)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,ult_fec_cli_1t,indrel_1mes,tiprel_1mes,indresi,indext,conyuemp,canal_entrada,indfall,tipodom,cod_prov,nomprov,ind_actividad_cliente,renta,segmento,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0,6,1,NaN,1.0,A,S,N,NaN,KHL,N,1,29,MALAGA,1,"87,218",02 - PARTICULARES,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,S,NaN,KHE,N,1,13,CIUDAD REAL,0,"35,549",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHE,N,1,13,CIUDAD REAL,0,"122,179",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHD,N,1,50,ZARAGOZA,0,"119,776",03 - UNIVERSITARIO,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0,35,1,NaN,1,A,S,N,NaN,KHE,N,1,50,ZARAGOZA,1,NaN,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Базовая предобработка

In [ ]:
# Даты
date_cols = ["fecha_dato", "fecha_alta", "ult_fec_cli_1t"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")


# Продуктовые признаки (таргет/фичи)
product_cols = [c for c in df.columns if c.endswith("_ult1")]

for col in product_cols:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .fillna(0)
        .clip(0, 1)   # защита от мусора
        .astype(np.uint8)
    )


# Числовые признаки
num_cols = [
    "age",
    "antiguedad",
    "renta",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# Дополнительная очистка числовых
# возраст
if "age" in df.columns:
    df["age"] = df["age"].clip(18, 100)

# стаж
if "antiguedad" in df.columns:
    df["antiguedad"] = df["antiguedad"].clip(0, 500)

# доход (очень важный признак)
if "renta" in df.columns:
    df["renta"] = df["renta"].clip(0, df["renta"].quantile(0.99))
    df["renta_log"] = np.log1p(df["renta"])


# Категориальные признаки
cat_cols = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")


# Сортировка по времени
df = df.sort_values(["ncodpers", "fecha_dato"]).reset_index(drop=True)

# Формирование таргета

In [ ]:
# сортировка по клиенту и времени
df = df.sort_values(["ncodpers", "fecha_dato"]).copy()

# предыдущий продуктовый портфель клиента
prev_products = df.groupby("ncodpers")[product_cols].shift(1)

# признаки предыдущего состояния
prev_product_cols = [f"prev_{col}" for col in product_cols]
prev_products_for_features = prev_products.fillna(0).astype(np.uint8).copy()
prev_products_for_features.columns = prev_product_cols

# таргет: только новые продукты
target_df = (df[product_cols] - prev_products.fillna(0)).clip(lower=0).astype(np.uint8)
target_cols = [f"target_{col}" for col in product_cols]
target_df.columns = target_cols

# объединяем
df = pd.concat([df, prev_products_for_features, target_df], axis=1)

# удаляем первое наблюдение клиента, у которого нет истории
df["row_num_per_client"] = df.groupby("ncodpers").cumcount()
df = df[df["row_num_per_client"] > 0].copy()

print(df.shape)
print("Количество target-признаков:", len(target_cols))
print("Количество prev-признаков:", len(prev_product_cols))
print("NaN в таргете:", df[target_cols].isna().sum().sum())

In [ ]:
# распределение таргета
df["new_products"] = df[target_cols].sum(axis=1)

print(df["new_products"].describe())
print(df["new_products"].value_counts(normalize=True).sort_index().head(10))

In [ ]:
# самые частые новые продукты
display(df[target_cols].sum().sort_values(ascending=False).head(10))

# Feature engineering

In [ ]:
# агрегированные признаки
df["num_products"] = df[product_cols].sum(axis=1).astype(np.uint8)

df["tenure_days"] = (df["fecha_dato"] - df["fecha_alta"]).dt.days
df["tenure_days"] = df["tenure_days"].clip(lower=0)
df["tenure_days"] = df["tenure_days"].fillna(df["tenure_days"].median())

df["month"] = df["fecha_dato"].dt.month.astype(np.uint8)

# списки признаков
num_features = [
    "age",
    "antiguedad",
    "renta",
    "renta_log",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
    "num_products",
    "tenure_days",
    "month",
]

cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

# используем только prev_* как продуктовые признаки
feature_cols = num_features + cat_features + prev_product_cols

print("Количество числовых признаков:", len(num_features))
print("Количество категориальных признаков:", len(cat_features))
print("Количество prev-признаков:", len(prev_product_cols))
print("Итого признаков:", len(feature_cols))

# проверка на leakage
leak_cols = [c for c in feature_cols if c in product_cols]
print("Leakage product_cols in feature_cols:", leak_cols)

## Обработка категорий

### Для LightGBM можно закодировать категории через category.

In [ ]:
# 1. Числовые признаки
num_features = [
    "age",
    "antiguedad",
    "renta",
    "renta_log",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
    "num_products",
    "tenure_days",
    "month",
]

num_features = [c for c in num_features if c in df.columns]

for col in num_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# базовая очистка
if "age" in df.columns:
    df["age"] = df["age"].clip(18, 100)

if "antiguedad" in df.columns:
    df["antiguedad"] = df["antiguedad"].clip(0, 500)

if "tenure_days" in df.columns:
    df["tenure_days"] = df["tenure_days"].clip(lower=0)

if "renta" in df.columns:
    upper_renta = df["renta"].quantile(0.99)
    df["renta"] = df["renta"].clip(lower=0, upper=upper_renta)

if "renta_log" not in df.columns and "renta" in df.columns:
    df["renta_log"] = np.log1p(df["renta"])

# медианная импутация
for col in num_features:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())


# 2. Категориальные признаки
cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

cat_features = [c for c in cat_features if c in df.columns]

for col in cat_features:
    df[col] = df[col].astype(str).replace("nan", "UNKNOWN").astype("category")


# 3. Финальный список признаков

feature_cols = num_features + cat_features + prev_product_cols

print("Количество числовых признаков:", len(num_features))
print("Количество категориальных признаков:", len(cat_features))
print("Количество prev-признаков:", len(prev_product_cols))
print("Итого признаков:", len(feature_cols))

print("\nПример числовых признаков:")
print(num_features[:10])

print("\nПример категориальных признаков:")
print(cat_features[:10])

print("\nПример prev-признаков:")
print(prev_product_cols[:5])

In [ ]:
# функция оптимизации памяти
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory before: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        # пропускаем даты
        if pd.api.types.is_datetime64_any_dtype(col):
            continue

        # числовые
        if pd.api.types.is_numeric_dtype(col):
            c_min = df[col].min()
            c_max = df[col].max()

            if pd.api.types.is_integer_dtype(col):
                if c_min >= 0:
                    if c_max < 255:
                        df[col] = df[col].astype(np.uint8)
                    elif c_max < 65535:
                        df[col] = df[col].astype(np.uint16)
                    else:
                        df[col] = df[col].astype(np.uint32)
                else:
                    df[col] = df[col].astype(np.int32)

            else:
                df[col] = df[col].astype(np.float32)

        # строки → category
        elif df[col].dtype == "object":
            num_unique = df[col].nunique()
            num_total = len(df[col])

            if num_unique / num_total < 0.5:
                df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory after: {end_mem:.2f} MB")
    print(f"Reduced by: {(start_mem - end_mem) / start_mem * 100:.1f}%")

    return df

In [ ]:
# оптимизируем память
df = reduce_memory_usage(df)

## Time-based split

In [ ]:
train_df = df[df["fecha_dato"] < "2016-05-28"].copy()
valid_df = df[df["fecha_dato"] == "2016-05-28"].copy()

# ФИЛЬТР — оставляем только строки с покупками
train_df["new_products"] = train_df[target_cols].sum(axis=1)

print("До фильтра:", train_df.shape)

train_df = train_df[train_df["new_products"] > 0].copy()

print("После фильтра:", train_df.shape)
print("Train positive rate:", train_df[target_cols].sum().sum() / len(train_df))

### Почему разбиваем именно так?

* Это последний месяц в train-данных, 2016-05-28 — это последняя дата этого месяца.

* Мы моделируем будущее. Ближайшая дата к “будущему”

## Подготовка матриц признаков

In [ ]:
X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

for col in cat_features:
    X_train[col] = X_train[col].astype(str).replace("nan", "UNKNOWN").astype("category")
    X_valid[col] = X_valid[col].astype(str).replace("nan", "UNKNOWN").astype("category")

for col in num_features:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_valid[col] = pd.to_numeric(X_valid[col], errors="coerce")

    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_valid[col] = X_valid[col].fillna(median_value)

print(X_train.shape, X_valid.shape)

# Обучене модели 

## Baseline Top Popular

### Считаем самые популярные новые продукты на train:

In [ ]:
popular_targets = train_df[target_cols].sum().sort_values(ascending=False)
top_popular_products = [c.replace("target_", "") for c in popular_targets.index.tolist()]
top_popular_products[:5]

## Метрики

In [ ]:
def precision_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    return len(set(actual) & set(predicted)) / k if k > 0 else 0.0


def recall_at_k(actual, predicted, k=5):
    predicted = predicted[:k]
    return len(set(actual) & set(predicted)) / len(actual) if len(actual) > 0 else 0.0


def apk(actual, predicted, k=5):
    predicted = predicted[:k]
    score = 0.0
    hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            hits += 1.0
            score += hits / (i + 1.0)

    return score / min(len(actual), k) if len(actual) > 0 else 0.0


def mean_metrics(actual_list, predicted_list, k=5):
    precisions = [precision_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    recalls = [recall_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    maps = [apk(a, p, k) for a, p in zip(actual_list, predicted_list)]

    return {
        f"precision_at_{k}": float(np.mean(precisions)),
        f"recall_at_{k}": float(np.mean(recalls)),
        f"map_at_{k}": float(np.mean(maps)),
    }

## Подготовка фактических ответов для valid

In [ ]:
# Actual для validation

def extract_actual_products(row, target_cols):
    return [c.replace("target_", "") for c in target_cols if row[c] == 1]

valid_actual = valid_df[target_cols].apply(
    lambda row: extract_actual_products(row, target_cols),
    axis=1
).tolist()

print("valid rows:", len(valid_actual))
print("non-empty actual:", sum(len(x) > 0 for x in valid_actual))
print("share non-empty actual:", round(sum(len(x) > 0 for x in valid_actual) / len(valid_actual), 4))

## Baseline Top Popular

In [ ]:
K = 5

popular_targets = train_df[target_cols].sum().sort_values(ascending=False)
top_popular_products = [c.replace("target_", "") for c in popular_targets.index.tolist()]

valid_pred_top_pop = [top_popular_products[:K] for _ in range(len(valid_df))]

baseline_metrics = mean_metrics(valid_actual, valid_pred_top_pop, k=K)
print("Baseline metrics (all):", baseline_metrics)

mask_non_empty = [len(x) > 0 for x in valid_actual]
valid_actual_pos = [a for a, m in zip(valid_actual, mask_non_empty) if m]
valid_pred_top_pop_pos = [p for p, m in zip(valid_pred_top_pop, mask_non_empty) if m]

print("Baseline metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=K))

## Обучение LightGBM one-vs-rest

### по одной модели на каждый продукт.

In [ ]:
models = {}
valid_scores = pd.DataFrame(index=valid_df.index)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
    "is_unbalance": True,
}

MIN_POSITIVES = 300

for i, target_col in enumerate(target_cols, start=1):
    y_train = train_df[target_col].astype(int)
    y_valid = valid_df[target_col].astype(int)

    n_pos_train = int(y_train.sum())
    n_pos_valid = int(y_valid.sum())

    print(
        f"[{i}/{len(target_cols)}] {target_col} | "
        f"train positives={n_pos_train}, valid positives={n_pos_valid}"
    )

    if n_pos_train < MIN_POSITIVES:
        print(f"  -> skip: too few positives in train (< {MIN_POSITIVES})")
        valid_scores[target_col] = 0.0
        continue

    if y_train.nunique() < 2:
        print("  -> skip: only one class in train")
        valid_scores[target_col] = 0.0
        continue

    train_dataset = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_features,
        free_raw_data=False
    )

    valid_dataset = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=cat_features,
        free_raw_data=False
    )

    pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

    params_local = params.copy()
    params_local["scale_pos_weight"] = pos_weight

    model = lgb.train(
        params_local,
        train_dataset,
        valid_sets=[valid_dataset],
        num_boost_round=300,
    )

    models[target_col] = model
    valid_scores[target_col] = model.predict(
        X_valid,
        num_iteration=model.best_iteration
    )

print(f"\nОбучено моделей: {len(models)} из {len(target_cols)}")
print("Форма valid_scores:", valid_scores.shape)

## Формирование рекомендаций модели

### Важно не рекомендовать уже имеющиеся у клиента продукты.

In [ ]:
def get_model_recommendations(
    valid_df,
    score_df,
    product_cols,
    prev_product_cols,
    top_k=5,
    top_popular_products=None
):
    recommendations = []

    prev_to_product = {f"prev_{p}": p for p in product_cols}

    for idx in valid_df.index:
        # ИСКЛЮЧАЕМ только продукты, которые уже были до текущего месяца
        existing_products = set(
            prev_to_product[col]
            for col in prev_product_cols
            if valid_df.loc[idx, col] == 1
        )

        scores = {
            target_col.replace("target_", ""): score_df.loc[idx, target_col]
            for target_col in score_df.columns
        }

        filtered_scores = {
            p: s for p, s in scores.items() if p not in existing_products
        }

        ranked = sorted(filtered_scores.items(), key=lambda x: x[1], reverse=True)
        recs = [p for p, _ in ranked[:top_k]]

        if top_popular_products is not None and len(recs) < top_k:
            for p in top_popular_products:
                if p not in recs and p not in existing_products:
                    recs.append(p)
                if len(recs) == top_k:
                    break

        recommendations.append(recs)

    return recommendations

## Оценка LightGBM

In [ ]:
valid_pred_lgb = get_model_recommendations(
    valid_df=valid_df,
    score_df=valid_scores,
    product_cols=product_cols,
    prev_product_cols=prev_product_cols,
    top_k=K,
    top_popular_products=top_popular_products
)

lgb_metrics = mean_metrics(valid_actual, valid_pred_lgb, k=K)
print("LightGBM metrics (all):", lgb_metrics)

valid_pred_lgb_pos = [p for p, m in zip(valid_pred_lgb, mask_non_empty) if m]
print("LightGBM metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_lgb_pos, k=K))

## Сравнение моделей

In [ ]:
# функция для табличного стравнения моделей
def add_model_result(
    results_df: pd.DataFrame,
    model_name: str,
    metrics: dict,
    extra_info: dict | None = None
) -> pd.DataFrame:
    """
    Добавляет результаты модели в таблицу сравнения.

    Parameters
    ----------
    results_df : pd.DataFrame
        Текущая таблица результатов.
    model_name : str
        Название модели.
    metrics : dict
        Словарь метрик, например:
        {"precision_at_5": 0.01, "recall_at_5": 0.03, "map_at_5": 0.02}
    extra_info : dict | None
        Дополнительная информация, например:
        {"split": "valid", "notes": "with prev features"}

    Returns
    -------
    pd.DataFrame
        Обновлённая таблица результатов.
    """
    row = {"model": model_name}
    row.update(metrics)

    if extra_info is not None:
        row.update(extra_info)

    row_df = pd.DataFrame([row])

    if results_df is None or results_df.empty:
        return row_df

    return pd.concat([results_df, row_df], ignore_index=True)

In [ ]:
comparison_df = pd.DataFrame()

comparison_df = add_model_result(
    comparison_df,
    model_name="Top Popular",
    metrics=baseline_metrics,
    extra_info={"evaluation": "all_rows"}
)

comparison_df = add_model_result(
    comparison_df,
    model_name="LightGBM",
    metrics=lgb_metrics,
    extra_info={"evaluation": "all_rows"}
)

comparison_df = add_model_result(
    comparison_df,
    model_name="Top Popular",
    metrics=mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=5),
    extra_info={"evaluation": "positive_rows_only"}
)

comparison_df = add_model_result(
    comparison_df,
    model_name="LightGBM",
    metrics=mean_metrics(valid_actual_pos, valid_pred_lgb_pos, k=5),
    extra_info={"evaluation": "positive_rows_only"}
)

comparison_df = comparison_df.sort_values(["evaluation", "map_at_5"], ascending=[True, False]).reset_index(drop=True)
display(comparison_df)

## Логирование в MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("bank_product_recommendation")

with mlflow.start_run(run_name="lightgbm_one_vs_rest_v1"):

    # параметры
    mlflow.log_param("model_type", "lightgbm_one_vs_rest")
    mlflow.log_param("top_k", K)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("num_products", len(product_cols))

    # baseline метрики
    for metric_name, metric_value in baseline_metrics.items():
        mlflow.log_metric(f"baseline_{metric_name}", float(metric_value))

    # модельные метрики
    for metric_name, metric_value in lgb_metrics.items():
        mlflow.log_metric(metric_name, float(metric_value))

print("MLflow logging completed")

## Сохранение модели

In [ ]:
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# pkl
joblib.dump(models, os.path.join(MODELS_DIR, "lgbm_one_vs_rest.pkl"))

# bin
for target_col, model in models.items():
    model.save_model(os.path.join(MODELS_DIR, f"{target_col}.bin"))

print("Модели сохранены")

In [ ]:
print("TRAIN target sum:", train_df[target_cols].sum().sum())
print("VALID target sum:", valid_df[target_cols].sum().sum())

valid_actual = valid_df[target_cols].apply(
    lambda row: [c.replace("target_", "") for c in target_cols if row[c] == 1],
    axis=1
).tolist()

num_non_empty_actual = sum(len(x) > 0 for x in valid_actual)
print("Non-empty actual in valid:", num_non_empty_actual)
print("Share:", round(num_non_empty_actual / len(valid_actual), 4))

print("\nTop valid targets:")
display(valid_df[target_cols].sum().sort_values(ascending=False).head(10))

print("\nScore stats:")
display(valid_scores.describe())

print("\nExamples:")
for i in range(5):
    print("ACTUAL:", valid_actual[i])
    print("PRED  :", valid_pred_lgb[i])
    print("-" * 40)

In [ ]:
# 1. actual
valid_actual = valid_df[target_cols].apply(
    lambda row: [c.replace("target_", "") for c in target_cols if row[c] == 1],
    axis=1
).tolist()

# 2. метрики по всем
print("Baseline metrics (all):", mean_metrics(valid_actual, valid_pred_top_pop, k=5))
print("LightGBM metrics (all):", mean_metrics(valid_actual, valid_pred_lgb, k=5))

# 3. метрики только по строкам с положительными событиями
mask_non_empty = [len(x) > 0 for x in valid_actual]

valid_actual_pos = [a for a, m in zip(valid_actual, mask_non_empty) if m]
valid_pred_top_pop_pos = [p for p, m in zip(valid_pred_top_pop, mask_non_empty) if m]
valid_pred_lgb_pos = [p for p, m in zip(valid_pred_lgb, mask_non_empty) if m]

print("Baseline metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=5))
print("LightGBM metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_lgb_pos, k=5))

# 4. сколько таких строк
print("Positive rows in valid:", sum(mask_non_empty), "out of", len(mask_non_empty))